In [69]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [70]:
training_data100 = datasets.CIFAR100(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

test_data100 = datasets.CIFAR100(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

c:\Users\Krzychu\Desktop\Informatyka\SN_PROJEKT\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [71]:
start_batch_size = 512
batch_schedule = {
    4:  256,
    7:  64,
    15: 32,
}

def get_train_loader(batch_size):
    return DataLoader(training_data100, batch_size=batch_size, shuffle=True)

def get_test_loader(batch_size):
    return DataLoader(test_data100, batch_size=batch_size)

train_dataloader100 = get_train_loader(start_batch_size)
test_dataloader100  = get_test_loader(start_batch_size)

In [72]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

Using cuda device


In [73]:
# Define model
class ConvNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.3),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.4),
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

          

    def forward(self, x):
        return self.classifier(self.features(x))

model100 = ConvNet(num_classes=100).to(device)
print(model100)

ConvNet(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout(p=0.2, inplace=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU(inplace=True)
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, cei

In [ ]:
# Adadelta  - najgorsze ze wszystkich
# Adafactor - kiepskie
# Adagrad   - kiepskie
# Adam      - dobrze, ale Adam W. lepszy
# AdamW     - chyba najlepszy
# Adamax Implements Adamax algorithm (a variant of Adam based on infinity norm).
# NAdam - niezle, ale ADAM ciutke lepiej
# RAdam - niezle, ale ADAM ciutke lepiej
# Muon   -  kiepsko
# RMSprop - niezle, ale < ADAM W.
# Rprop - kiepskie
# SGD stochastic gradient descent                           dziala kiepsko
# ASGD Implements Averaged Stochastic Gradient Descent.     skoro SGD dziala kiepsko to nie probowalbym tego

# lista schedulerow:
# StepLR — redukuje lr o stala wartosc co x epok
# MultiStepLR — jak StepLR, ale redukcja w okreslonych z gory epokach (np. 10., 15., 20.)
# ExponentialLR — nie zaglebiami sie w dzialaniu, plynniejszy niz poprzednie
# ConstantLR — raz zmniejsza/zwieksza, np. 0.01 0.01 0.01 0.005 0.005 0.005
# LinearLR — nieciekawe
# PolynomialLR - zmnienia wielomianowo
# CosineAnnealingLR — obniza lr wzdluz krzywej cosinus do minimum, potem restart
# CosineAnnealingWarmRestarts — jak wyzej, ale z automatycznymi restartami co n epok gdyby nie osiagnelo minimum
# CyclicLR
# OneCycleLR
# ReduceLROnPlateau — redukuje lr gdy loss (przy mode=min) nie spada lub accuracy nie rosnie
loss_fn = nn.CrossEntropyLoss()
start_lr=0.0015
optimizer100 = torch.optim.RAdam(model100.parameters(), lr=start_lr)
from torch.optim.lr_scheduler import ReduceLROnPlateau
scheduler = ReduceLROnPlateau(
    optimizer100,
    mode='min',        
    factor=0.1,        
    patience=1,        
    min_lr=5e-07,
    threshold_mode='abs',    
    threshold=0.025,   
    cooldown=1        
)

In [75]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss_value, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss_value:>7f}  [{current:>5d}/{size:>5d}]")

In [76]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    accuracy = 100 * correct
    print(f"Test Error: \n Accuracy: {accuracy:>0.1f}%, Avg loss: {test_loss:>8f} \n")
    return accuracy, test_loss

In [77]:
epochs = 60 
training_log = []

for t in range(epochs):
    print(f"Epoch {t+1} (CIFAR-100)\n-------------------------------")
    
    if(t in batch_schedule):
        current_batch_size = batch_schedule[t]
        train_dataloader100 = get_train_loader(current_batch_size)
        test_dataloader100  = get_test_loader(current_batch_size)
        print("Batch size zmieniony na ",current_batch_size)

    train(train_dataloader100, model100, loss_fn, optimizer100)
    accuracy, avg_loss = test(test_dataloader100, model100, loss_fn)
    training_log.append({
        "epoch": t + 1,
        "accuracy": accuracy,
        "avg_loss": avg_loss,
    })
    lr_before = optimizer100.param_groups[0]['lr']
    scheduler.step(avg_loss)
    lr_after = optimizer100.param_groups[0]['lr']

    if lr_after != lr_before:
        print(f"Epoch {t+1}: lr zmienione {lr_before:.6f} → {lr_after:.6f}")
    else:
        print(f"Epoch {t+1}: lr = {lr_after:.6f}")
print("Done CIFAR-100")

Epoch 1 (CIFAR-100)
-------------------------------


loss: 4.790759  [  512/50000]
Test Error: 
 Accuracy: 19.0%, Avg loss: 3.361495 

Epoch 1: lr = 0.001500
Epoch 2 (CIFAR-100)
-------------------------------
loss: 3.339700  [  512/50000]
Test Error: 
 Accuracy: 20.2%, Avg loss: 3.407576 

Epoch 2: lr = 0.001500
Epoch 3 (CIFAR-100)
-------------------------------
loss: 2.875339  [  512/50000]
Test Error: 
 Accuracy: 34.0%, Avg loss: 2.535818 

Epoch 3: lr = 0.001500
Epoch 4 (CIFAR-100)
-------------------------------
loss: 2.556999  [  512/50000]
Test Error: 
 Accuracy: 36.9%, Avg loss: 2.442483 

Epoch 4: lr = 0.001500
Epoch 5 (CIFAR-100)
-------------------------------
Batch size zmieniony na  256
loss: 2.159551  [  256/50000]
loss: 2.563860  [25856/50000]
Test Error: 
 Accuracy: 37.0%, Avg loss: 2.444242 

Epoch 5: lr = 0.001500
Epoch 6 (CIFAR-100)
-------------------------------
loss: 2.407394  [  256/50000]
loss: 2.156002  [25856/50000]
Test Error: 
 Accuracy: 37.8%, Avg loss: 2.389085 

Epoch 6: lr = 0.001500
Epoch 7 (CIFAR-100)
-

KeyboardInterrupt: 

In [78]:
import json
import os

models_dir = "models100"
meta_dir = "metadata100"

# sprawdzamy ile jest plikow w folderze
n = len([name for name in os.listdir(models_dir) if os.path.isfile(os.path.join(models_dir, name))])
file_index = n + 1

# jesli jest n modeli, to nowy bedzie cifar100_(n+1)
model_path = os.path.join(models_dir, f"cifar100_{file_index}.pth")
torch.save(model100.state_dict(), model_path)

training_metadata = {
    "network_architecture": str(model100),
    "optimizer": optimizer100.__class__.__name__,
    "learning_rate": start_lr,
    "scheduler": scheduler.__class__.__name__,
    "scheduler_params": {
        "mode": scheduler.mode,
        "factor": scheduler.factor,
        "patience": scheduler.patience,
        "min_lr": scheduler.min_lrs[0] if hasattr(scheduler, "min_lrs") else None,
        "threshold": scheduler.threshold,
        "cooldown": scheduler.cooldown,
    },
    "batch_size": start_batch_size,
    "training_log": training_log,
}

metadata_path = os.path.join(meta_dir, f"{file_index}.txt")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(training_metadata, f, ensure_ascii=False, indent=2)

print(f"Found {n} files in {models_dir}.")
print(f"Saved model to {model_path}")
print(f"Saved training metadata to {metadata_path}")

Found 11 files in models100.
Saved model to models100\cifar100_12.pth
Saved training metadata to metadata100\12.txt
